# Air Quality Monitoring Using Official & Community Sensors

**Student Name: Saheel Ahmed (3357368)**  

**Course name: Acquisition & Exploration of GeoData (2024-1B)**

**Assignment: 1**

**Month, Year: January, 2025**

## Introduction

Air pollution, particularly particulate matter like PM₂.₅ and PM₁₀, poses significant health risks, contributing to respiratory and cardiovascular diseases and reducing life quality (U.S. EPA, 2024). While air quality has improved in Europe, air pollution remains a leading environmental health hazard, especially in cities like The Hague, where pollution levels often exceed safe thresholds (EEA, 2024; IQAir, 2025). Monitoring air quality is crucial for effective intervention, but traditional stations have limited coverage, especially in densely populated areas. This notebook explores air quality data from both official and community-based sensors in The Hague, evaluating their reliability and optimizing sensor placement to improve coverage, with a focus on schools.

## Data Acquisition

### Imports

In [ ]:
%%writefile requirements.txt
geopandas
pandas
numpy
osmnx
requests
seaborn

In [ ]:
!pip3 install -r requirements.txt

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import osmnx
import requests
import json
import seaborn
import shapely
%matplotlib inline

### Admin Boundary

The code retrieves the polygon for The Hague, Netherlands, using the OSM Turbo Overpass API.

In [ ]:
import osmnx as ox
import geopandas as gpd

# OSM Relation ID for the shape the Hague
# https://www.openstreetmap.org/relation/192737
relation_id = 192737

# Download the shape of the relation from OpenStreetMap
gdf = ox.geometries_from_place(f"relation/{relation_id}", tags={})
gdf = gdf.to_crs("EPSG:28992")
# Save it as a GeoJSON or shapefile if needed
output_path = "the_hague.geojson"
gdf.to_file(output_path, driver="GeoJSON")

print(f"Shape successfully downloaded and saved to {output_path}")

# Display basic info about the data
print(gdf.head())

### Official Air Quality Sensors

The code fetches all official sensors within The Hague by requesting data from the Luchtmeetstations API, then downloads the data and saves it as a GeoJSON file.

In [ ]:
import requests
import geopandas as gpd
from shapely.geometry import shape

def fetch_stations_name(municipality_name):
    # Code to retrieve station name & coordinates from stations api
    base_url = "https://api.luchtmeetnet.nl/open_api/stations"
    params = {"page": 1}

    aq_stations = []

    while True:
        response = requests.get(base_url, params=params)

        if response.status_code != 200:
            print(f"Error: {response.status_code}")
            break

        data = response.json()
        stations = data.get("data", [])
        pagination = data.get("pagination", {})

        # Filter stations by municipality name (case-insensitive)
        aq_stations.extend([station for station in stations if municipality_name in station.get("location", "").lower()])

        # Check if there's a next page
        if pagination.get("next_page") == pagination.get("current_page"):
            break
        params["page"] += 1

    return aq_stations

def fetch_stations_coordinates(station_code):
    # Code to retrieve stations coordinates
    base_url = f"https://api.luchtmeetnet.nl/open_api/stations/{station_code}"
    response = requests.get(base_url)
    if response.status_code == 200:
        return response.json()["data"]
    raise Exception(f"Request failed with status code {response.status_code}")

if __name__ == "__main__":
    MUNICIPALITY_NAME = "den haag"
    aq_stations = fetch_stations_name(MUNICIPALITY_NAME)
    print(f"Air Quality stations within '{MUNICIPALITY_NAME}':", aq_stations)

    # Fetch coordinates for each station
    aq_stations_coordinates = [fetch_stations_coordinates(station['number']) for station in aq_stations]

    # Create GeoDataFrame and save to file
    aq_gdf = gpd.GeoDataFrame(aq_stations_coordinates)
    aq_gdf["geometry"] = aq_gdf["geometry"].apply(lambda x: shape(x))
    aq_gdf.set_geometry("geometry", crs="EPSG:4326", inplace=True)
    aq_gdf = aq_gdf.to_crs("EPSG:28992")
    aq_gdf.to_file("den-haag_luchtmeetnet-official-sensors.geojson", driver="GeoJSON")


### Schools

The code retrieves all schools within The Hague using the OSM Turbo Overpass API and saves the data as a GeoJSON file.

In [ ]:
import osmnx as ox
import geopandas as gpd

# Load boundary and set CRS
boundary_gdf = gpd.read_file("den-haag.geojson").to_crs("EPSG:28992")

# Query OSM for schools within the boundary
schools = ox.features.features_from_polygon(boundary_gdf.unary_union, {"amenity": "school"})

# Save schools to a GeoDataFrame and clip by boundary
schools_gdf = gpd.GeoDataFrame(schools, crs="EPSG:4326").to_crs(boundary_gdf.crs)
schools_gdf = gpd.clip(schools_gdf, boundary_gdf)

# Process and save the results
schools_gdf["geometry"] = schools_gdf.geometry.apply(lambda geom: geom.centroid if geom.type == "Polygon" else geom)
schools_gdf = schools_gdf[["name", "amenity", "id", "geometry"]]
schools_gdf = schools_gdf.to_crs("EPSG:28992")
schools_gdf.to_file("schools_within_den-haag.geojson")
print("Schools saved to schools_within_den-haag.geojson")

### Community Air Quality Sensors

The code retrieves all community sensors from the Samen Meten API and filters the sensors to include only those located within The Hague.

In [ ]:
import requests

# Initial API endpoint
url = "https://api-samenmeten.rivm.nl/v1.0/Locations"

# API request headers
headers = {
    "Accept": "application/json",
    "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:132.0) Gecko/20100101 Firefox/132.0"
}

# List to store all results
all_results = []

while url:
    # Make the API request with headers
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        raise Exception(f"Failed request with status code {response.status_code}: {response.text}")

    # Parse the JSON response
    data = response.json()

    # Append results to the list
    all_results.extend(data.get("value", []))

    # Get the next link, or None if it's not present
    url = data.get("@iot.nextLink")

# Now `all_results` contains all the data from paginated responses
print(f"Total results collected: {len(all_results)}")

# Save list of dictionaries as JSON
output_file = "samenmeten-community-sensors.json"

with open(output_file, "w") as file:
    json.dump(all_results, file, indent=2)

print(f"Data successfully saved to {output_file}")

Select or clip sensors located within the Den Haag boundary.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, shape

# Load the shapefile (boundary)
boundary_gdf = gpd.read_file("den-haag.geojson")

# Load all community sensors
with open("samenmeten-community-sensors.json", "r") as fout:
  community_sensors = json.load(fout)

# Create Point geometries using Shapely
geometry = [Point(item["location"]["coordinates"]) for item in community_sensors]

# Convert the list of dictionaries into a GeoDataFrame
data = gpd.GeoDataFrame(all_results, geometry=geometry, crs="EPSG:4326")

# Filter points within the shapefile boundary
filtered_data = data[data.within(boundary_gdf.unary_union)]

# Save filtered result as geodataframe
filtered_data.to_file("den-haag_samenmeten-community-sensors.geojson")

## Data Processing

### Filter community stations closest to each official stations

The code identifies the closest community sensors to the official sensors by using a 500m buffer (later expanded to 1.5 km) to search for nearby community sensors.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import itertools

# Load GeoJSON files
pollutant = "pm25"
gdf_official = gpd.read_file(f"all-official-sensors-{pollutant}.geojson")
gdf_community = gpd.read_file(f"all-community-sensors-{pollutant}.geojson")

NL_CRS = "EPSG:28992"
# Ensure both GeoDataFrames use the same CRS
gdf_official = gdf_official.to_crs(NL_CRS)
gdf_community = gdf_community.to_crs(NL_CRS)

# Create a dictionary to store the results
closest_stations = {}

# Define the maximum distance (500 mean 500 meters)
max_distance = 500

# Iterate through each station in official
for index_a, station_a in gdf_official.iterrows():
    # Get the geometry of the station in official
    geom_a = station_a.geometry

    # Find all stations in community within 500m of the current station in official
    nearby_stations = gdf_community[gdf_community.geometry.distance(geom_a) <= max_distance]

    # Store the results in the dictionary
    closest_stations[station_a["location"]] = list(nearby_stations["name"])

closest_stations_list = set(list(itertools.chain.from_iterable(closest_stations.values())))
# Filter the closest community sensors from all community sensors
gdf_community_closest = gdf_community[gdf_community['name'].isin(closest_stations_list)]
gdf_community_closest.to_file("selected-community-sensors.geojson")

### Downloading air quality data from each community sensors

The code first retrieves all the links to datastreams associated with the selected community sensors. Each datastream is a URL containing all measurement data for a specific sensor. It then iterates through these datastreams page by page to download the complete measurement data for the entire time period.

In [ ]:
import geopandas as gpd
import requests
import pandas as pd

# Load the GeoDataFrame
gdf = gpd.read_file("selected-community-sensors.geojson")

# Initialize lists for extracted data
pm25_kal_data = []
pm10_kal_data = []

# Iterate through each row in the GeoDataFrame
for _, row in gdf.iterrows():
    things_url = row["Things@iot.navigationLink"]

    try:
        things_response = requests.get(things_url)
        things_response.raise_for_status()
        things_data = things_response.json()

        # Get Datastreams URL from the Things data
        datastreams_url = things_data.get("value", [{}])[0].get("Datastreams@iot.navigationLink")
        if datastreams_url:
            datastreams_response = requests.get(datastreams_url)
            datastreams_response.raise_for_status()
            datastreams_data = datastreams_response.json()

            for datastream in datastreams_data.get("value", []):
                if "name" in datastream:
                    name = datastream["name"].lower()
                    observation_url = datastream.get("Observations@iot.navigationLink")

                    if observation_url:
                        # Append data to the appropriate list based on "name"
                        if "pm25_kal" in name:
                            pm25_kal_data.append({
                                "sensor_id": row["@iot.id"],
                                "observation_link": observation_url,
                                "parent_name": row["parent_name"]
                            })
                        elif "pm10_kal" in name:
                            pm10_kal_data.append({
                                "sensor_id": row["@iot.id"],
                                "observation_link": observation_url,
                                "parent_name": row["parent_name"]
                            })
    except requests.exceptions.RequestException as e:
        print(f"Error processing URL {things_url}: {e}")

# Convert to DataFrames and save to CSV
pd.DataFrame(pm25_kal_data).to_csv("community-pm25-datastream.csv", index=False)
pd.DataFrame(pm10_kal_data).to_csv("community-pm10-datastream.csv", index=False)

# Display the first few rows for each DataFrame
print("PM10 Data:", pd.DataFrame(pm10_kal_data).head())
print("PM25 Data:", pd.DataFrame(pm25_kal_data).head())

Fetch Measurements

In [ ]:
import requests
import pandas as pd
import datetime

# API request headers (modify as needed)
headers = {
    "Accept": "application/json",  # Ensure API returns JSON
    "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:132.0) Gecko/20100101 Firefox/132.0"
}
current_datetime = datetime.datetime.utcnow()
end_datetime = datetime.datetime.fromisoformat("2020-12-31T00:00:00")
duration_in_sec = (current_datetime - end_datetime).total_seconds()
num_observations = divmod(duration_in_sec, 3600)[0]
pollutant = "pm25"
gdf = pd.read_csv(f"community-{pollutant}-datastream.csv")

def fetch_observations(num_observations, base_url):
  observations = []
  observations_to_skip = 0
  observations_to_select = 200
  while len(observations) <= num_observations:
    params = {
        "$top": observations_to_select,
        "$skip": observations_to_skip
    }
    response = requests.get(base_url, params=params, headers=headers)
    response.raise_for_status()
    data = response.json()
    items = data.get("value", [])
    if not items:
      break
    observations.extend(items)
    observations_to_skip += observations_to_select
  print(f"Total items fetched from {base_url}: {len(observations)}")
  return observations

# Main logic to iterate through the DataFrame
combined_data = {}
import uuid
for index, row in gdf.iterrows():
    link = row['observation_link']
    station_id = row['sensor_id']

    # Fetch data for this link
    data = fetch_observations(num_observations, link)
    # Save the data as a dictionary with the station_id as the key
    combined_data[str(station_id)] = data

# Final result
print(f"Fetched data for {len(combined_data)} stations.")

# Convert combined_data to a DataFrame
rows = []

for sensor_id, records in combined_data.items():
    for record in records:
        row = {"name": sensor_id, **record}
        rows.append(row)

# Create a DataFrame
combined_df = pd.DataFrame(rows)
combined_df.to_csv(f"community-{pollutant}-measurements.csv", encoding="utf-8", index=False)

### Combining Community and Official Air Quality Measurements

The code integrates community sensor data with the closest official sensor data. It begins by mapping the IDs of community and official sensors that are in close proximity. A common key (official sensor ID) is then added to each community sensor's data. The data is aggregated from hourly to daily averages and merged using the common key and date.

In [ ]:
# Load community and official measurements
pollutant = "pm10"
df_community = pd.read_csv(f"community-{pollutant}-measurements.csv")
df_official = pd.read_csv(f"official-{pollutant}-measurements.csv")

# Add Official sensor ID to the Community dataset
sensors_mapping = {
    "pm25": {
    30828: "NL10450",
    29783: "NL10450",
    30848: "NL10404",
    30819: "NL10404"
    },
    "pm10": {
    29987: "NL10450",
    29405: "NL10404",
    22905: "NL10404",
    13871: "NL10445",
    12234: "NL10445"
    }
}

# Map the ID's
df_community["meetlocatie_id"] = df_community["name"].apply(lambda x: sensor_mapping[pollutant][x])
# Format the date
df_community["date"] = pd.to_datetime(df_community["phenomenonTime"], format='mixed', utc=True).dt.date
df_official["date"] = pd.to_datetime(df_official["begindatumtijd"], format='mixed', utc=True).dt.date
# Average the measurements
df_official_agg = df_official.groupby(["meetlocatie_id", "date"]).agg({
    "waarde": "mean"
}).reset_index()
df_community_agg = df_community.groupby(['date', "name"]).agg({
    "result": "mean",
}).reset_index()
# Rename the measurements column
df_community_agg.rename({
    "result": f"{pollutant}_community"
}, axis = 1, inplace=True)

df_official_agg.rename({
    "waarde": f"{pollutant}_official"
}, axis=1, inplace=True)
# Merge the official and community measurements
df_combined = pd.merge(df_community_agg, df_official_agg, on=["meetlocatie_id", "date"], how="inner")

## Plotting Offical & Community Measurements

Scatter Plot

The code plots a scatter plot between community and official sensors, displaying the correlation between their measurements using the Pearson correlation coefficient and the error metric RMSE.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

# Key and value for the sensors
community = 30848
official = "NL10404"
pollutant_label = "PM10"

fig, ax = plt.subplots(1, 1, figsize=(16, 10))
# Filter the data for the specific sensors
single_off_sensor = df_combined[
    (df_combined["name"] == community) & (df_combined["meetlocatie_id"] == official)
].copy()

# Drop rows with missing values
single_off_sensor = single_off_sensor.dropna()

pearson_corr, p_value = pearsonr(single_off_sensor[f"{pollutant}_official"], single_off_sensor[f"{pollutant}_community"])
print("Pearson Correlation Coefficient:", pearson_corr)
print("P-value:", p_value)

# Plot the data
sns.scatterplot(data=single_off_sensor, x=f"{pollutant}_community", y=f"{pollutant}_official", ax=ax, s=100)
# Add a diagonal reference line
ax.plot([0, 80], [0, 80], color='red', linestyle='--', label='x = y')
# Add R^2 text to the plot
ax.text(1, 70, f"$PCC(r)$: {pearson_corr:.2f} ", fontsize=20, color='#0377fc', )
ax.text(1, 65, f"$P-Value(p<0.001)$: {p_value:.2e} ", fontsize=20, color='#0377fc')
ax.text(1, 60, f"$N$: {single_off_sensor.shape[0]} ", fontsize=20, color='#0377fc')
# Set plot limits and labels
plt.xlim(0, 80)
plt.ylim(0, 80)
plt.xlabel(f"Community Sensor {community} {pollutant_label} (ug/m3)")
plt.ylabel(f"Official Sensor {official} {pollutant_label} (ug/m3)")
plt.title(f"Comparing Official Sensor ({official}) with Community ({community}) for {pollutant_label} (ug/m3)")
plt.legend()
# Save the figure
fig.savefig(f'{official}-{community}-{pollutant}.png', dpi=fig.dpi, bbox_inches='tight', pad_inches=0.1)
plt.show();

Time Series Chart

The code plots with time series graph showing periods of aggrement and drift between community and official sensors.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

# Create the figure and axes
fig, ax = plt.subplots(1, 1, figsize=(18, 10))

single_off_sensor[f'{pollutant}_official'] = single_off_sensor[f"{pollutant}_official"].rolling(window=7).mean()
single_off_sensor[f'{pollutant}_community'] = single_off_sensor[f"{pollutant}_community"].rolling(window=7).mean()
single_off_sensor["date"] = pd.to_datetime(single_off_sensor["date"])

# Plot the 'result' column
sns.lineplot(data=single_off_sensor, x='date', y=f'{pollutant}_community', ax=ax, label=f'Community {pollutant_label}', linewidth = 1.5)

# Plot the 'PM25' column
sns.lineplot(data=single_off_sensor, x='date', y=f'{pollutant}_official', ax=ax, label=f'Official {pollutant_label}', linewidth = 1.5)

# Add titles and labels
ax.set_title(f"Comparison of Offical Sensor ({official}) with Community ({community}) For {pollutant_label} (ug/m3) Over Time (7 Day Rolling Mean)", fontsize=16)
ax.set_xlabel("Date", fontsize=14)
ax.set_ylabel(f"{pollutant_label} (ug/m3)", fontsize=14)

# Format the x-axis ticks
ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=10))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))

# Rotate xticks for better readability
plt.xticks(rotation=45, ha='right')

# Add a legend
ax.legend(title="Legend", fontsize=12, title_fontsize=12)

# Show the plot
plt.show()
fig.savefig(f'{official}-{community}-{pollutant}-line.png', dpi=fig.dpi, bbox_inches='tight', pad_inches=0.1);

## Defining "reliability"

The code defines reliability using a decision-based weighted sum of parameters. It calculates reliability based on four key parameters: the correlation factor, root mean square error (RMSE), number of observations, and the distance between community and official sensors.

In [ ]:
# Normalization parameters
n_min, n_max = 30, 365*2       # Min and max for number of observations
d_min, d_max = 0.1, 2          # Min and max for distance
rmse_min, rmse_max = 1, 20      # Min and max for MSE (adjust based on your data)

# Weights for the factors
w1, w2, w3, w4 = 0.3, 0.2, 0.2, 0.3

def get_reliability_index(pearson_corr, distance, num_observations, rmse):
    # Normalize factors
    norm_pcc = (pearson_corr + 1) / 2
    norm_n = (num_observations - n_min) / (n_max - n_min)
    norm_d = 1 - (distance - d_min) / (d_max - d_min)
    norm_mse = 1 - (rmse - rmse_min) / (rmse_max - rmse_min)

    # Ensure normalized values are within [0, 1]
    norm_n = max(0, min(1, norm_n))
    norm_d = max(0, min(1, norm_d))
    norm_mse = max(0, min(1, norm_mse))

    # Calculate reliability index
    reliability_index = (
        w1 * norm_pcc +
        w2 * norm_n +
        w3 * norm_d +
        w4 * norm_mse
    )
    reliability_index = max(0, min(1, reliability_index))  # Ensure RI is between 0 and 1
    return reliability_index

# Apply the function to the dataframe
metrics_df["Reliability_Index"] = metrics_df.apply(
    lambda x: get_reliability_index(
        x["r(PCC)"],
        x["Distance_from_Official_Sensor"],
        x["Number_of_Observation"],
        x["RMSE"]
    ),
    axis=1
)

## Plotting Reliability vs Distance Chart

The code plots a reliability vs. distance chart, with RMSE represented by color and correlation shown as the size proportion of the points.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Define a function for the inverse relationship
def inverse_relationship(x, a, b):
    return a / x + b

metrics_df = pd.read_csv("reliability_metrics.csv")
# Extract x (distance) and y (reliability index) data
x = metrics_df["Distance_from_Official_Sensor"][:-1]
y = metrics_df["Reliability_Index"][:-1]

# Fit the inverse relationship function to the data
popt, _ = curve_fit(inverse_relationship, x, y)

# Generate fitted curve data
x_curve = np.linspace(min(x), max(x), 500)
y_curve = inverse_relationship(x_curve, *popt)

# Bubble Chart
plt.figure(figsize=(10, 6))
bubble_chart = plt.scatter(
    metrics_df["Distance_from_Official_Sensor"],
    metrics_df["Reliability_Index"],
    s=metrics_df["Number_of_Observation"],  # Bubble size
    c=metrics_df["RMSE"],  # Bubble color
    cmap='coolwarm',
    alpha=0.7,
    edgecolor='k'
)

# Add the fitted curve
plt.plot(x_curve, y_curve, color='red', linestyle='--', label=f'Reliability vs Distance Curve (Inverse)\n RI = {popt[0]:.4f} / Distance + {popt[1]:.4f}')

# Add colorbar for r(PCC)
cbar = plt.colorbar(bubble_chart)
cbar.set_label("RMSE (ug/m3)", fontsize=10, fontweight="bold")


# Add labels for each bubble
for i, row in metrics_df.iterrows():
    plt.text(
        row["Distance_from_Official_Sensor"],  # X-coordinate
        row["Reliability_Index"]- 0.005,  # Y-coordinate
        row["Label"],  # The label from the 'Label' column
        fontsize=6,  # Font size
        ha='left',  # Horizontal alignment
        va='top',  # Vertical alignment
        bbox=dict(boxstyle="round,pad=0.3", edgecolor='gray', facecolor='white', alpha=0.7)  # Optional bounding box
    )


# Add legend for bubble sizes
bubble_sizes = [50, 200, 500, 700]  # Example observation counts
for size in bubble_sizes:
    plt.scatter([], [], s=size, c='gray', alpha=0.5, label=f'{size} Observations')

outlier = metrics_df.iloc[-1]
# Draw an arrow and label for the outlier
plt.annotate(
    "Excluded Outlier",
    xy=(outlier["Distance_from_Official_Sensor"], outlier["Reliability_Index"]),
    xytext=(outlier["Distance_from_Official_Sensor"] + 5, outlier["Reliability_Index"] + 0.02),
    arrowprops=dict(facecolor='red', arrowstyle='->'),
    fontsize=10,
    color="red"
)

# Add legend for the curve
plt.legend(scatterpoints=1, frameon=True, labelspacing=1, title="Legend")

# Labels and title
plt.title("Reliability Index vs Distance", fontsize=14, fontweight="bold")
plt.xlabel("Distance from Official Sensor (m)", fontsize=12, fontweight="bold")
plt.ylabel("Reliability Index", fontsize=12, fontweight="bold")

# Show the plot
plt.grid(True, linestyle="--", alpha=0.6)
plt.savefig("rel-vs-distance.png",bbox_inches='tight', dpi=300)
plt.show();

## Defining "Placement Strategy"

The code defines a placement strategy for installing new official and community sensors. The strategy is based on a weighted approach similar to reliability scores, utilizing parameters such as the number of schools, population, and the distance to the closest official sensors.

A spatial join is performed between the schools and population data to integrate the relevant spatial information for sensor placement.

In [ ]:
import geopandas as gpd

# Load GeoDataFrames
official = gpd.read_file("official-sensors.geojson")
schools = gpd.read_file("schools.geojson")
population = gpd.read_file("population.geojson")

# Spatial join to count schools within population areas
population_with_schools = gpd.sjoin(schools, population[["geometry"]], predicate='within') \
    .groupby("index_right").size() \
    .reset_index(name="school_count")

# Add school count back to population GeoDataFrame
population_with_schools = population.join(population_with_schools.set_index("index_right")["school_count"], how="left")

# Calculate nearest sensor distance and normalize
population_with_schools['Nearest_Sensor_Distance'] = population_with_schools.geometry.centroid.apply(
    lambda x: official.distance(x).min()
)
d_min, d_max = population_with_schools['Nearest_Sensor_Distance'].min(), population_with_schools['Nearest_Sensor_Distance'].max()
population_with_schools['Distance_Norm'] = (population_with_schools['Nearest_Sensor_Distance'] - d_min) / (d_max - d_min)

# Normalize school count and population
population_with_schools['School_Norm'] = (population_with_schools['school_count'] - population_with_schools['school_count'].min()) / (population_with_schools['school_count'].max() - population_with_schools['school_count'].min())
population_with_schools['Population_Norm'] = (population_with_schools['population'] - population_with_schools['population'].min()) / (population_with_schools['population'].max() - population_with_schools['population'].min())

# Define weights and calculate composite score
w_s, w_p, w_d = 0.5, 0.3, 0.2
population_with_schools['Composite_Score'] = w_s * population_with_schools['School_Norm'] + w_p * population_with_schools['Population_Norm'] + w_d * population_with_schools['Distance_Norm']

## Placement of new sensors based on composite placement score

The code generates a map displaying all newly installed official and community sensors, based on the calculated composite placement score.

In [ ]:
import numpy as np

def placement_category(value):
    if value > 0.48:
        return "Official"
    elif value >= 0.3 and value <= 0.48:
        return "Community"
    else:
        return np.nan

population_with_schools["Placement_Category"] = population_with_schools["Composite_Score"].apply(lambda x: placement_category(x))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

colors = {
    "Official": "blue",
    "Community": "green",
}

cmap = ListedColormap(list(colors.values()))

# Plot the hex grid
fig, ax = plt.subplots(figsize=(10, 8))

# Plot cells with valid Composite_Score
valid_cells = population_with_schools[population_with_schools['Composite_Score'].notna()]
valid_cells.plot(column='Composite_Score', cmap='YlOrRd', legend=True, ax=ax, edgecolor='black')

# Plot cells with NaN Composite_Score using a hatched pattern
nan_cells = population_with_schools[population_with_schools['Composite_Score'].isna()]
nan_cells.plot(ax=ax, color='none', edgecolor='black', hatch='///', label='_nolegend_')

valid_cells_placement = population_with_schools[population_with_schools['Composite_Score'].notna()]
valid_cells_placement["geometry"] = valid_cells_placement.geometry.centroid
valid_cells_placement.plot(column='Placement_Category', categorical=True, cmap=cmap, legend=True, ax=ax, edgecolor='black')

# Add a title and show the plot
plt.title("Optimal Air Quality Sensor Placement with Composite Scores", fontsize=14)
plt.grid(True)
plt.show()

## Calculation of Reliability Score post network expansion

The code calculates the reliability of the entire network after the installation of the new sensors.

In [ ]:
import geopandas as gpd


gdf = gpd.read_file("all_sensors.geojson")

official_sensors = gdf[gdf["Placement_Category"]=="Official"].copy()

# Calculate the closest distance to an official sensor for each community sensor
def calculate_min_distance(community_sensor, official_sensors):
    # Calculate distance from this community sensor to all official sensors
    distances = official_sensors.distance(community_sensor.geometry)
    return distances.min()

def get_new_rel_score(x):
  return 0.613 + 0.20 * (1 - ((x - min_distance) / (max_distance - min_distance)))

# Apply the function to all community sensors
gdf['min_distance_to_official'] = gdf.apply(
    lambda row: calculate_min_distance(row, official_sensors), axis=1
)

gdf["reliability_score"] = gdf.apply(lambda x: get_new_rel_score(x["min_distance_to_official"]) if x["reliability_score"]==0.613 else x["reliability_score"], axis=1)